# Submissão 3B 

In [ ]:
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

sys.path.append('..')
from src.data_utils import encode_texts
from src.pytorch_models import select_device

device = select_device()
print('Device:', device)

In [ ]:
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, num_classes, dropout,
                 embedding_matrix=None, freeze=False, bidirectional=False):
        super().__init__()
        if embedding_matrix is not None:
            self.embedding = nn.Embedding.from_pretrained(
                embedding_matrix, freeze=freeze, padding_idx=0
            )
        else:
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * (2 if bidirectional else 1), num_classes),
        )

    def forward(self, x):
        x = self.embedding(x)
        _, h_n = self.gru(x)
        last_hidden = h_n[-1]
        return self.classifier(last_hidden)

In [ ]:
artifact = torch.load('../modelos/gru_fasttext.pt', map_location=device, weights_only=False)

vocab        = artifact['vocab']
max_len      = artifact['max_len']
class_order  = artifact['class_order']
label_to_idx = artifact['label_to_idx']
idx_to_label = {v: k for k, v in label_to_idx.items()}
cfg          = artifact['config']

print('Classes:', class_order)
print('Config:', cfg)

In [ ]:
model = GRUClassifier(
    vocab_size=len(vocab),
    embed_dim=cfg['embed_dim'],
    hidden_dim=cfg['hidden_dim'],
    num_layers=cfg['num_layers'],
    num_classes=len(class_order),
    dropout=cfg['dropout'],
).to(device)

model.load_state_dict(artifact['state_dict'])
model.eval()
print('Modelo carregado.')

In [ ]:
df_subm = pd.read_csv('../data/subm3.csv', sep=';')
print(f'Textos a classificar: {len(df_subm)}')
df_subm.head()

In [ ]:
X = encode_texts(df_subm['Text'].values, vocab, max_len)
x_tensor = torch.tensor(X, dtype=torch.long, device=device)

with torch.no_grad():
    logits = model(x_tensor)
    pred_idx = logits.argmax(dim=1).cpu().tolist()

pred_labels = [idx_to_label[i] for i in pred_idx]

df_out = pd.DataFrame({'ID': df_subm['ID'], 'Labels': pred_labels})

print('Distribuição de labels:')
print(df_out['Labels'].value_counts())
df_out.head()

In [ ]:
output_path = 'subm3-g1-MEI-B.csv'
df_out.to_csv(output_path, index=False, sep=';')
print('Guardado:', output_path)